This notebook trains and applies the classifiers. For each feature set (AEF, OlmoEarth, both), it predicts the crop of every test field with two standard ways of evaluating embeddings: a linear probe (multinomial logistic regression), which tests whether straight boundaries separate the crops, and k-nearest neighbours (kNN), which tests whether a field's nearest neighbours share its crop. Each method's one setting (the probe's penalty, kNN's k) is chosen on the validation fields. Notebook 04 plots the results.

In [ ]:
import os
import time
import copy
import warnings
import numpy as np
import pandas as pd
import pyarrow as pa
from pathlib import Path
from sklearn.metrics import f1_score,log_loss
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings('ignore',category=ConvergenceWarning)

In [ ]:
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
FEATURESETS = {'aef':[f'aef{i:03d}' for i in range(128)],
               'olmoearth':[f'olmoearth{i:03d}' for i in range(128)],
               'both':[f'aef{i:03d}' for i in range(128)]+[f'olmoearth{i:03d}' for i in range(128)]}
TRAINFIELDS = 20000
VALFIELDS   = 5000
PENALTIES   = [1,10,100,1000]
KS          = [5,10,25,50]
STEPITERS   = 10
MAXSTEPS    = 50
PATIENCE    = 5
SEED        = 42
NCORES      = len(os.sched_getaffinity(0))

pa.set_cpu_count(NCORES)
pa.set_io_thread_count(NCORES)

`TRAINFIELDS` and `VALFIELDS` are the fields sampled from each country's training and validation blocks. Every test field is predicted. `PENALTIES` are the L2 penalty strengths tried for the probe (a larger penalty keeps the weights smaller), and `KS` are the neighbour counts tried for kNN. The probe is fitted in steps of `STEPITERS` optimiser iterations, up to `MAXSTEPS` steps, and stops once the validation loss has not improved for `PATIENCE` steps.

In [ ]:
def load_collection(collection):
    table  = pd.read_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet")
    blocks = table.chip.str[4:-4]+'_'+(table.chip.str[-4:-2].astype(int)//6).astype(str)+'_'+(table.chip.str[-2:].astype(int)//6).astype(str)
    mixed  = int((table.groupby(blocks).split.nunique()>1).sum())
    sample = pd.concat([table[table.split=='train'].sample(min(TRAINFIELDS,(table.split=='train').sum()),random_state=SEED),
                        table[table.split=='val'].sample(min(VALFIELDS,(table.split=='val').sum()),random_state=SEED),
                        table[table.split=='test']])
    return sample.assign(country=collection.split('/')[1]),{'blocks':blocks.nunique(),'mixedblocks':mixed}

def fit_probe(featureset,penalty):
    model   = LogisticRegression(C=1/penalty,class_weight='balanced',max_iter=STEPITERS,warm_start=True)
    history,best,beststep = [],None,0
    for step in range(1,MAXSTEPS+1):
        model.fit(xtrain[featureset],train.crop)
        trainloss = log_loss(train.crop,model.predict_proba(xtrain[featureset]),labels=model.classes_,sample_weight=trainweights)
        valloss   = log_loss(val.crop,model.predict_proba(xval[featureset]),labels=model.classes_,sample_weight=valweights)
        history.append({'featureset':featureset,'penalty':penalty,'iterations':step*STEPITERS,'trainloss':trainloss,'valloss':valloss})
        if valloss<=min(row['valloss'] for row in history):
            best,beststep = copy.deepcopy(model),step
        if step-beststep>=PATIENCE:
            break
    return best,history

def vote(distances,neighbours,k):
    scores = np.zeros((len(neighbours),len(classes)))
    np.add.at(scores,(np.arange(len(neighbours))[:,None],traincodes[neighbours[:,:k]]),1/(distances[:,:k]+1e-9))
    return classes[scores.argmax(axis=1)]

In [ ]:
loaded = [load_collection(collection) for collection in COLLECTIONS]
fields = pd.concat([sample for sample,_ in loaded],ignore_index=True)
blocks = pd.DataFrame([check for _,check in loaded])
train  = fields[fields.split=='train']
val    = fields[(fields.split=='val')&fields.crop.isin(train.crop.unique())]
test   = fields.loc[fields.split=='test',['id','country','crop']].copy()

scalers      = {featureset:StandardScaler().fit(train[cols]) for featureset,cols in FEATURESETS.items()}
xtrain       = {featureset:scalers[featureset].transform(train[cols]) for featureset,cols in FEATURESETS.items()}
xval         = {featureset:scalers[featureset].transform(val[cols]) for featureset,cols in FEATURESETS.items()}
xtest        = {featureset:scalers[featureset].transform(fields.loc[fields.split=='test',cols]) for featureset,cols in FEATURESETS.items()}
trainweights = compute_sample_weight('balanced',train.crop)
valweights   = compute_sample_weight('balanced',val.crop)
classes,traincodes = np.unique(train.crop,return_inverse=True)

print(f'Loaded {len(fields):,} fields from {fields.country.nunique()} countries...')
print(f'- {len(train):,} training, {len(val):,} validation and {len(test):,} test fields, {len(classes)} crops')
print(f'- {blocks.mixedblocks.sum()} of {blocks.blocks.sum():,} 6 km blocks have fields in more than one split (should be 0)')

## Linear probe

First choose the penalty: fit the probe on both feature sets with each value in `PENALTIES` and keep the one with the lowest validation loss. Then fit one probe per feature set with that penalty, recording the training and validation loss as it fits, and predict the test fields.

In [ ]:
sweep   = {penalty:fit_probe('both',penalty)[1] for penalty in PENALTIES}
bestval = {penalty:min(row['valloss'] for row in history) for penalty,history in sweep.items()}
PENALTY = min(bestval,key=bestval.get)

print('Lowest validation loss for each penalty (both feature sets)...')
for penalty,loss in bestval.items():
    print(f'- penalty {penalty:,}: {loss:.3f}')
print(f'Using penalty {PENALTY:,}')

In [ ]:
histories = [row for history in sweep.values() for row in history]
for featureset in FEATURESETS:
    start         = time.perf_counter()
    model,history = fit_probe(featureset,PENALTY)
    test[f'probe_{featureset}'] = model.predict(xtest[featureset])
    histories    += history
    minutes,seconds = divmod(round(time.perf_counter()-start),60)
    print(f"- probe {featureset}: test accuracy {(test.crop==test[f'probe_{featureset}']).mean():.3f} ({minutes}m {seconds}s)")

## k-nearest neighbours

For each feature set, find every validation field's nearest training fields once, score each k in `KS` on the validation fields by macro F1, and keep the best. Votes are weighted by inverse distance, so closer neighbours count more. Then predict the test fields with that k.

In [ ]:
knnsweep = []
for featureset in FEATURESETS:
    start      = time.perf_counter()
    index      = NearestNeighbors(n_neighbors=max(KS),algorithm='brute',n_jobs=NCORES).fit(xtrain[featureset])
    distances,neighbours = index.kneighbors(xval[featureset])
    scores     = {k:f1_score(val.crop,vote(distances,neighbours,k),average='macro') for k in KS}
    bestk      = max(scores,key=scores.get)
    distances,neighbours = index.kneighbors(xtest[featureset],n_neighbors=bestk)
    test[f'knn_{featureset}'] = vote(distances,neighbours,bestk)
    knnsweep  += [{'featureset':featureset,'k':k,'valmacrof1':score} for k,score in scores.items()]
    minutes,seconds = divmod(round(time.perf_counter()-start),60)
    print(f"- kNN {featureset}: best k {bestk}, test accuracy {(test.crop==test[f'knn_{featureset}']).mean():.3f} ({minutes}m {seconds}s)")

In [ ]:
test.to_parquet(PROCDIR/'results_predictions.parquet',index=False)
pd.DataFrame(histories).to_csv(PROCDIR/'results_probelosses.csv',index=False)
pd.DataFrame(knnsweep).to_csv(PROCDIR/'results_knnsweep.csv',index=False)

print('Saved for notebook 04...')
print(f"- {PROCDIR/'results_predictions.parquet'}: the true crop and 6 predictions (2 methods × 3 feature sets) for every test field")
print(f"- {PROCDIR/'results_probelosses.csv'}: probe training and validation loss by iteration, for the penalty sweep and the final fits")
print(f"- {PROCDIR/'results_knnsweep.csv'}: kNN validation macro F1 for each k")